# Nevergrad comparison

Used for comparing Nevergrad optimisers.  Since CMA-ES was selected can also run for different locations, values of sigma and locations.

In [1]:
import nevergrad as ng
import numpy as np
import time
from pathlib import Path

from modules.helper_functions_general import (
    read_index,
)

from modules.helper_functions_tsp import (
    find_problem_size,
    find_distances_array,
    cost_fn_fact,
    vqc_circuit,
    define_parameters,
    create_initial_rotations,
    find_sdk,
    calculate_hot_start_data,
    find_device_string,
    find_run_stats,
    validate_qubit_loops,
    find_valid_device_loop,
    find_qubit_loop_fidelity,
)

from modules.helper_functions_nevergrad import (
    ng_cost_function_fact,
)

from classes.MyDataLogger import MyDataLogger, MySubDataLogger

from modules.config import (
    CONTROL_DIR,
    CONTROL_FILE,
    ENCODING,
    VALID_QUBIT_LOOPS,
    TARGET_FIDELITY,
    PRINT_FREQUENCY,
)

from braket.jobs import hybrid_job, save_job_result
from braket.jobs.metrics import log_metric

PRODUCTION_RUN = True # test input first

In [2]:
control_path = Path(CONTROL_DIR).joinpath(CONTROL_FILE)
control_dict = read_index(control_path, ENCODING)
print(f'Reading control data from {control_path}')
print(control_dict)
EMPTY = ' '

Reading control data from control\control_parameters.csv
{0: {'quantum': 'TRUE', 'locations': '10', 'slice': '1', 'shots': '1024', 'mode': '21', 'iterations': '1250', 'gray': 'FALSE', 'hot_start': 'TRUE', 'gradient_type': 'CMA', 'formulation': 'original', 'layers': '1', 'std_dev': '', 'lr': '', 'weight_decay': '', 'momentum': ' ', 'alpha': '', 'big_a': '', 'c': '', 'eta': '', 'gamma': '', 's': '', 'noise': 'FALSE', 'mps': 'FALSE', 'aws': 'TRUE', 'target': 'cepheus', 'sigma': '0.7'}}


In [3]:
len_dict = len(control_dict)
if len_dict != 1:
    raise Exception(
        f'The control dictionary has {len_dict} items, only one should be submitted to a quantum device currently'
    )

In [4]:
def apply_notebook_specific_validation(sdl):
    """Cepheus specific checks """
    if sdl.target != 'cepheus':
        raise Exception('This code is only current written for submission to the Cepheus device')
    if sdl.slice != 1:
        raise ValueError(f'Please run with slice = 1, not slice = {sdl.slice}')
    if sdl.layers !=1:
        raise ValueError('fPlease run with one layer, not {sdl.layers}')
    if sdl.gradient_type != 'CMA':
        raise ValueError(f'this notebook is only written to process CMA, not {sdl.gradient_type=}')
    if sdl.sigma != 0.7:
        raise ValueError(f'This notebook is only written to process CMA with sigma = 0.7, not {sdl.sigma=}')
    if not(sdl.aws):
        raise Exception('please run for AWS only')
    if sdl.mps:
        raise Exception('MPS is not supported')
    if sdl.noise:
        raise Exception('Simulating with noise is not supported (or required)')
    if sdl.alpha or sdl.big_a or sdl.c or sdl.eta or sdl.gamma or sdl.s:
        raise ValueError('Please do not set the SPSA parameters')
    if sdl.std_dev or sdl.lr or sdl.momentum:
        raise ValueError('Please do not set the ML parameters')
    if sdl.gray:
        raise ValueError('Not tested with gray code')
    if sdl.formulation != 'original':
        raise ValueError(f'Not tested with {sdl.formulation=}')
    return

Instantiate datalogger

In [5]:
datalogger = MyDataLogger()
t0 = time.time()
data_dict = control_dict[0]
sdl = MySubDataLogger(runid=datalogger.runid)
sdl.update_constants_from_dict(data_dict)
sdl.validate_input()
print(f'{sdl.sigma=}')
apply_notebook_specific_validation(sdl)
device_arn = find_device_string(sdl.target)
print(f'Running for {sdl.target=} with {device_arn}')
sdk_type = find_sdk(sdl.target)
print(f'Key settings are: {sdl.aws=}, {sdk_type=}')
print(f'Also {sdl.mps=}, {sdl.slice=},')
sdl.qubits = find_problem_size(locations=sdl.locations, formulation=sdl.formulation)
print(
    f'There are {sdl.qubits} logical qubits needed for {sdl.locations} locations in the {sdl.formulation} formulation.'
)
print(f'Running mode {sdl.mode} with {sdl.shots} shots')

num_params = sdl.calculate_parameter_numbers()
print(f'Number of parameters to be optimized is {num_params}')

distance_array, sdl.best_dist = find_distances_array(sdl.locations, print_comments=True)

print(f'The best distance is known to be {sdl.best_dist}')

SubDataLogger instantiated.  Run ID = 20260710-18-43-43 - 18-43-43
sdl.sigma=0.7
Running for sdl.target='cepheus' with arn:aws:braket:us-west-1::device/qpu/rigetti/Cepheus-1-108Q
Key settings are: sdl.aws=True, sdk_type='aws'
Also sdl.mps=False, sdl.slice=1.0,
There are 21 logical qubits needed for 10 locations in the original formulation.
Running mode 21 with 1024 shots
num_params_per_qubit=2 qubits_measured=22 self.layers=1 
Number of parameters to be optimized is 44
Reading distance data
Data will be read from filename networks\sim_dist_10_locs.txt.
It is known that the shortest distance is 290.2
The best distance is known to be 290.2


In [6]:
print(f'Validating qubit loop for {sdl.target=} and {sdl.qubits=}')
qubit_loop = find_valid_device_loop(sdl.qubits, sdl.target)
print(f'Validating fidelity for {sdl.target=} and {sdl.qubits=}')
validate_qubit_loops(
    qubits=sdl.qubits, 
    loop_dict=VALID_QUBIT_LOOPS, 
    target=sdl.target
    )

total_error = find_qubit_loop_fidelity(
    qubits=sdl.qubits, 
    target=sdl.target
    )

if total_error < TARGET_FIDELITY:
    raise Exception(f'The total error {total_error:.3f} is higher than target of {TARGET_FIDELITY:.3f}')
else:
    print(f'The total error {total_error:.3f} is within tolerance')

Validating qubit loop for sdl.target='cepheus' and sdl.qubits=21
Validating fidelity for sdl.target='cepheus' and sdl.qubits=21
Found device as Device('name': Cepheus-1-108Q, 'arn': arn:aws:braket:us-west-1::device/qpu/rigetti/Cepheus-1-108Q)
No errors found for target='cepheus' qubits=21 

Found device as Device('name': Cepheus-1-108Q, 'arn': arn:aws:braket:us-west-1::device/qpu/rigetti/Cepheus-1-108Q)
For edge=16-17, error=0.9870: 
For edge=17-26, error=0.9940: 
For edge=25-26, error=0.9936: 
For edge=25-34, error=0.9924: 
For edge=34-43, error=0.9934: 
For edge=42-43, error=0.9774: 
For edge=33-42, error=0.9519: 
For edge=24-33, error=0.9833: 
For edge=23-24, error=0.9866: 
For edge=22-23, error=0.9952: 
For edge=22-31, error=0.9882: 
For edge=31-40, error=0.9910: 
For edge=39-40, error=0.9909: 
For edge=30-39, error=0.9892: 
For edge=21-30, error=0.9861: 
For edge=12-21, error=0.9767: 
For edge=12-13, error=0.9935: 
For edge=4-13, error=0.9899: 
For edge=4-5, error=0.9922: 
For edg

In [7]:
@hybrid_job(
    device=device_arn,  # needs to be a string
    include_modules=['modules', 'classes'],
    dependencies=[
        'graycode',
        'qiskit_ibm_runtime',
        'qiskit_aer',
        'torch',  # numpy, qiskit, braket, etc. are included by default
        'nevergrad',
    ],
)
def run_nevergrad(
    mode:int,
    num_params:int,
    target:str,
    qubits:int,
    noise_bool:bool,
    layers:int,
    locations:int,
    gray:bool,
    formulation:str,
    hot_start:bool,
    shots:int,
    mps:bool, 
    sigma:float,
    iterations:int,
    distance_array: np.ndarray,
    best_dist: float,
):
    print('Running with only CMA as the optimiser.')
    optimzer_function = ng.optimizers.CMA

    params = define_parameters(
        mode=mode,
        num_params=num_params,
        target=target,
    )

# Set up and print quantum circuit.
    qc = vqc_circuit(
        qubits=qubits,
        mode=mode,
        noise_bool=noise_bool,
        layers=layers,
        params=params,
        target=target,
    )
  
    # Set up the cost_fn using the factory.  cost_fn takes a bit string and returns a distance.
    cost_fn = cost_fn_fact(
        locations=locations,
        gray=gray,
        formulation=formulation,
        distance_array=distance_array,
    )

    bin_hot_start_list = []
    hot_start_distance = None
    if hot_start:
        bin_hot_start_list, hot_start_distance = calculate_hot_start_data(
            locations=locations,
            gray=gray,
            formulation=formulation,
            best_dist=best_dist,
            distance_array=distance_array,
            cost_fn=cost_fn,
            print_results=True,
                )
        print(f'(The hot start distance is {hot_start_distance} and the hot start list is {bin_hot_start_list}')

    # Set up the initial parameters, either from the hot start, or from random,
    init_rots = create_initial_rotations(
        qubits=qubits,
        num_params=num_params,
        target=target,
        hot_start=hot_start,
        bin_hot_start_list=bin_hot_start_list,
    )
    
    print(f'The initial parameters (weights) are {init_rots}')

    ng_cost_function = ng_cost_function_fact(
        qc=qc,
        target=target,
        noise_bool=noise_bool,
        shots=shots,
        cost_fn=cost_fn,
        mps=mps,
        params=params,
        init_rots=init_rots,
        qubits=qubits,
    )

    instrum = ng.p.Instrumentation(
    ng.p.Array(shape=(num_params,)).set_mutation(sigma=sigma)
)
    optimizer = optimzer_function(
    parametrization=instrum,
    budget=iterations,
                    )
    
    average_list, lowest_list, best_av_list, index_list = [], [], [], []

    best_av_to_date = float('inf')
    best_dist_found = float('inf')

    print(f'Running optmisation with budget of {optimizer.budget}')

    for i in range(optimizer.budget):
        index_list.append(i)
        candidate = optimizer.ask()
        x = candidate.args[0]
        value, lowest = ng_cost_function(x)
        optimizer.tell(candidate, value)
        average_list.append(value)
        best_av_to_date = min(best_av_to_date, value)
        best_dist_found = min(best_dist_found, lowest)
        best_av_list.append(best_av_to_date)
        lowest_list.append(best_dist_found)
        if i % PRINT_FREQUENCY == 0:
            print(
                f'In iteration {i} The best distance found to date is {best_dist_found:.3f}',
                flush=True
                )
            print(
                f'The average cost from the sample is {value:.3f} and the lowest value is {lowest:.3f}',
                flush=True,
            )
            # AWS hybrid job
            log_metric(
                metric_name='average_sample_cost', iteration_number=i, value=value
            )
            log_metric(
                metric_name='lowest_sample_cost', iteration_number=i, value=lowest
            )
            log_metric(
                metric_name='lowest_to_date',
                iteration_number=i,
                value=best_dist_found,
            )


    items, hits, misses = cost_fn.report_cache_stats()
  
    results = {
        'index_list': index_list,
        'average_list': average_list,
        'cost_list': average_list,
        'lowest_list': lowest_list,
        'best_dist_found': best_dist_found,
        'best_av_to_date': best_av_to_date,
        'last_av': value,
        'cache_items': items,
        'cache_hits': hits,
        'cache_misses': misses,
        'hot_start_dist': hot_start_distance,

    }

    print(results.keys())
    print(len(results["average_list"]))
    print(len(results["lowest_list"]))

    save_job_result(results)

    return results

In [8]:
def update_sub_data_logger(sdl, results):
    sdl.average_list_all, 
    sdl.index_list = results['index_list']
    sdl.average_list = results['average_list']
    sdl.lowest_list = results['lowest_list']
    sdl.sliced_list = results['cost_list']
    sdl.best_dist_found = results['best_dist_found']
    sdl.best_av_to_date = results['best_av_to_date']
    sdl.last_av = results['last_av']
    sdl.cache_items = results['cache_items']
    sdl.cache_hits = results['cache_hits']
    sdl.cache_misses = results['cache_misses']
    sdl.hot_start_dist = results['hot_start_dist']
    _, sdl.iteration_found = find_run_stats(sdl.lowest_list)
    sdl.save_detailed_results()
    sdl.save_results_to_csv()
    return ()

Call function module

In [ ]:
if PRODUCTION_RUN:
    job = run_nevergrad(
        mode=sdl.mode,
        num_params=num_params,
        target=sdl.target,
        qubits=sdl.qubits,
        noise_bool=sdl.noise,
        layers=sdl.layers,
        locations=sdl.locations,
        gray=sdl.gray,
        formulation=sdl.formulation,
        hot_start=sdl.hot_start,
        shots=sdl.shots,
        mps=sdl.mps,
        sigma=sdl.sigma,
        iterations=sdl.iterations,
        distance_array = distance_array,
        best_dist=sdl.best_dist
    )


        # A) Print off job statistics
    print('TYPE:', type(job))
    print('JOB ID:', getattr(job, 'id', 'NO-ID'))
    print('STATE:', job.state())  # CREATED | QUEUED | RUNNING | COMPLETED | FAILED

    # B) Check region for the submitting session using?
    import boto3
    print('DEFAULT REGION:', boto3.Session().region_name)

    # C) Double-check didn't accidentally enable local mode
    print(run_nevergrad.__name__)  # ensure object is bound and callable

    t1 = time.time()
    sdl.elapsed = t1 - t0

    update_sub_data_logger(sdl, job.result())

TYPE: <class 'braket.aws.aws_quantum_job.AwsQuantumJob'>
JOB ID: NO-ID
STATE: QUEUED
DEFAULT REGION: eu-west-2
run_nevergrad


In [ ]:
sdl.average_list

In [ ]:
if PRODUCTION_RUN:
    update_sub_data_logger(sdl, job.result())